# 07. ANFIS Training (MLP Surrogate)

### Objective
Train the MLP surrogate model on the 6-feature ANFIS input dataset and export weights for Unity deployment.

### I/O
- **Reads**: `../../data/processed/6_anfis_dataset.csv`
- **Writes**: `../../data/models/anfis_mlp_weights.json`
- **Writes**: `../../data/models/training_stats.json`
- **Deploys**: `anfis-demo-ui/models/`

In [1]:
import pandas as pd
import numpy as np
import json, os, shutil
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error

INPUT_PATH = '../../data/processed/6_anfis_dataset.csv'
MODEL_DIR  = '../../data/models'
os.makedirs(MODEL_DIR, exist_ok=True)

print('Libraries imported')

Libraries imported


In [2]:
df = pd.read_csv(INPUT_PATH)
print(f'Loaded {len(df)} rows')

Loaded 3240 rows


In [3]:
feature_cols = [
    'soft_combat', 'soft_collect', 'soft_explore',
    'delta_combat', 'delta_collect', 'delta_explore'
]

X = df[feature_cols].values
y = df['target_multiplier'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Training with {len(feature_cols)} features')
print(f'Train: {len(X_train)}  Test: {len(X_test)}')

Training with 6 features
Train: 2592  Test: 648


### Model architecture and approach chosen through two experiments:

**Experiment D** (`experiments/experiment_D_approach_comparison/Experiment_D_Complete.ipynb`)
compared direct formula, full ANFIS, and MLP surrogate. Full ANFIS requires 729 rule
evaluations per window and cannot port cleanly to Unity C# without scipy. The direct formula
is frozen at hand-coded coefficients. MLP runs as a matrix multiply in microseconds and
retrains on new data. That is why an MLP is used below.

**Experiment E** (`experiments/experiment_E_model_selection/Experiment_E_Complete.ipynb`)
benchmarked 9 regression models. MLP (16-8) was selected for the production pipeline:
compact at 257 parameters, fast inference, portable to plain C#, and achieves test MAE
and R² that meet the difficulty controller's accuracy requirements. The full model comparison
and architecture search results are in `experiment_F_anfis_ablation/outputs/mlp_architecture_search.csv`.

Because of both results, the pipeline uses `hidden_layer_sizes=(16, 8)` below.

In [4]:
print('Training MLP...')
model = MLPRegressor(
    hidden_layer_sizes=(16, 8),
    activation='relu',
    max_iter=500,
    random_state=42
)
model.fit(X_train, y_train)
print('Training complete')

Training MLP...
Training complete


In [5]:
y_train_pred = model.predict(X_train)
y_test_pred  = model.predict(X_test)

train_mae = mean_absolute_error(y_train, y_train_pred)
test_mae  = mean_absolute_error(y_test,  y_test_pred)
train_r2  = r2_score(y_train, y_train_pred)
test_r2   = r2_score(y_test,  y_test_pred)
train_mse = mean_squared_error(y_train, y_train_pred)
test_mse  = mean_squared_error(y_test,  y_test_pred)

# MLP output for a perfectly balanced, zero-delta player.
# The live engine maps this to display 1.0 (no difficulty change).
neutral_input = np.array([[1/3, 1/3, 1/3, 0, 0, 0]])
mlp_neutral   = float(model.predict(neutral_input)[0])

print('='*60)
print('TRAINING COMPLETE - OPTION B CANONICAL (v2.2)')
print('='*60)
print(f'Train R2:    {train_r2:.4f}')
print(f'Test  R2:    {test_r2:.4f}')
print(f'Train MAE:   {train_mae:.6f}')
print(f'Test  MAE:   {test_mae:.6f}')
print(f'MLP neutral: {mlp_neutral:.6f}  (balanced, zero-delta input)')
print('='*60)

TRAINING COMPLETE - OPTION B CANONICAL (v2.2)
Train R2:    0.8631
Test  R2:    0.9350
Train MAE:   0.014430
Test  MAE:   0.012322
MLP neutral: 0.931601  (balanced, zero-delta input)


In [6]:
# Export weights as JSON so the Unity C# engine can run the forward pass
# without any Python or ML library dependency.
export_data = {
    'architecture': {
        'input_size':    6,
        'hidden_layers': [16, 8],
        'output_size':   1,
        'activation':    'relu',
        'output_activation': 'linear'
    },
    'feature_order': feature_cols,
    'weights': [w.tolist() for w in model.coefs_],
    'biases':  [b.tolist() for b in model.intercepts_],
    'training_metrics': {
        'train_mae':      float(train_mae),
        'test_mae':       float(test_mae),
        'train_mse':      float(train_mse),
        'test_mse':       float(test_mse),
        'train_r2':       float(train_r2),
        'test_r2':        float(test_r2),
        'num_iterations': model.n_iter_,
        'num_samples':    len(X)
    },
    # The live engine applies: display = clamp(1.0 + (raw - mlp_neutral) * 2.0, 0.6, 1.4)
    # Recomputed here on every retrain so the JS calibration never needs updating.
    'mlp_neutral': round(mlp_neutral, 6),
    'version': 'v2.2-optionB-canonical',
    'status':  'ACTIVE'
}

WEIGHTS_PATH = os.path.join(MODEL_DIR, 'anfis_mlp_weights.json')
with open(WEIGHTS_PATH, 'w') as f:
    json.dump(export_data, f, indent=2)

print(f'Weights exported to {WEIGHTS_PATH}')

Weights exported to ../../data/models\anfis_mlp_weights.json


In [ ]:
# Deploy path derived from MODEL_DIR so it works on any machine without hardcoded paths.
# MODEL_DIR = '../../data/models' (relative to this notebook), so ../../../ reaches CollectGame.Model root.
DEPLOY_DIR = os.path.normpath(os.path.join(MODEL_DIR, '../../../anfis-demo-ui/models'))

FILES_TO_DEPLOY = ['anfis_mlp_weights.json', 'scaler_params.json', 'deployment_manifest.json']
try:
    os.makedirs(DEPLOY_DIR, exist_ok=True)
    for fname in FILES_TO_DEPLOY:
        src = os.path.join(MODEL_DIR, fname)
        if os.path.exists(src):
            shutil.copy2(src, DEPLOY_DIR)
            print(f'  [done] {fname} -> anfis-demo-ui/models/')
        else:
            print(f'  WARN: {fname} not found in data/models/')
except Exception as e:
    print(f'  Deploy skipped: {e}')

In [ ]:
# Save training stats for the demo UI to display
t = df['target_multiplier']
stats = {
    'num_samples': int(len(df)),
    'target_multiplier': {
        'min':  float(t.min()),
        'max':  float(t.max()),
        'mean': float(t.mean()),
        'std':  float(t.std())
    },
    'soft_membership_ranges': {
        col: [float(df[col].min()), float(df[col].max())]
        for col in ['soft_combat', 'soft_collect', 'soft_explore']
    },
    'mlp_neutral': round(mlp_neutral, 6)
}

STATS_PATH = os.path.join(MODEL_DIR, 'training_stats.json')
with open(STATS_PATH, 'w') as f:
    json.dump(stats, f, indent=2)
print(f'Training stats saved to {STATS_PATH}')

try:
    shutil.copy2(STATS_PATH, DEPLOY_DIR)
    print(f'  [done] training_stats.json -> anfis-demo-ui/models/')
except Exception:
    pass

print('\nTarget multiplier distribution:')
print(f'  min={t.min():.4f}  max={t.max():.4f}  mean={t.mean():.4f}  std={t.std():.4f}')
print(f'  mlp_neutral={mlp_neutral:.6f}')